# Bronze → Silver: cleanse raw tables
Python notebook (not PySpark). Reads raw (bronze) Delta tables, applies the rules in a JSON rules file, and writes cleansed (silver) tables.

**Per table, per run:** read bronze rows from source files not yet processed, in batches of about `BATCH_ROWS` rows (oldest files first) → cleanse columns → check allowed values (failures rejected, nulled or defaulted) → remove duplicates on the key columns → merge into the silver table (insert new keys, update existing), one commit per batch → record rejected rows, processed files and a run log.

- `TABLES = ""` processes every table in the rules file; name one or more to process only those.
- Runs are incremental: each bronze `_source_file` is processed once. `FULL_RELOAD = True` rebuilds silver tables from all bronze rows.
- Rejected rows go to `silver_rejects` with the reason and the original bronze row as JSON.

## Optional: pin library versions
Uncomment to use specific versions (must run before anything imports Polars; adds start-up time). Pin both packages together.

In [ ]:
# %pip install -q "polars==2.0.0" "deltalake==1.6.6"

In [ ]:
# Parameters (mark this cell as a parameter cell so a pipeline can override)
SOURCE_WORKSPACE = "MyWorkspace"
SOURCE_LAKEHOUSE = "ls_raw"          # bronze
SOURCE_SCHEMA    = "dbo"             # None if the Lakehouse is not schema-enabled
TARGET_WORKSPACE = ""                # "" = same as SOURCE_WORKSPACE
TARGET_LAKEHOUSE = "ls_cleansed"     # silver
TARGET_SCHEMA    = "dbo"
RULES_FILE       = "config/silver_rules.json"   # under Files/ in the TARGET Lakehouse

TABLES          = ""      # "" = every table in the rules file; or "customers_raw" / "customers_raw,orders_raw"
FULL_RELOAD     = False   # True = rebuild the silver table(s) from all bronze rows
DRY_RUN         = False   # True = apply the rules and report counts, write nothing
PARALLEL_TABLES = 1       # tables processed at once in this session
BATCH_ROWS      = 10_000_000  # new bronze files are processed in batches of about this many rows (a file is never split)
ROW_GROUP_ROWS  = 8_000_000

LOG_TABLE       = "silver_load_log"         # one row per table per run
REJECTS_TABLE   = "silver_rejects"          # one row per rejected value / row
PROCESSED_TABLE = "silver_processed_files"  # bronze source files already processed

In [ ]:
import gc, json, re, time, uuid
from datetime import date, datetime, timezone
from decimal import Decimal
from concurrent.futures import ThreadPoolExecutor
import polars as pl
import deltalake
import notebookutils
from deltalake import DeltaTable

print(f"polars {pl.__version__}, deltalake {deltalake.__version__}")

def collect_streaming(lf):
    """Run a lazy query with the streaming engine on any Polars version."""
    try:
        return lf.collect(engine="streaming")     # Polars 1.2x and later
    except (TypeError, ValueError):
        return lf.collect(streaming=True)         # older Polars

def _enc(name):
    return name.replace(" ", "%20")   # spaces in workspace/lakehouse names must be encoded

def lakehouse_base(workspace, lakehouse):
    return f"abfss://{_enc(workspace)}@onelake.dfs.fabric.microsoft.com/{_enc(lakehouse)}.Lakehouse"

SOURCE_BASE = lakehouse_base(SOURCE_WORKSPACE, SOURCE_LAKEHOUSE)
TARGET_BASE = lakehouse_base(TARGET_WORKSPACE or SOURCE_WORKSPACE, TARGET_LAKEHOUSE)

def storage_options():
    # fetched per call so long runs don't hit token expiry
    return {"bearer_token": notebookutils.credentials.getToken("storage"),
            "use_fabric_endpoint": "true"}

def tables_root(base, schema):
    return f"{base}/Tables/{schema}" if schema else f"{base}/Tables"

def source_table_path(name):
    return f"{tables_root(SOURCE_BASE, SOURCE_SCHEMA)}/{name}"

def target_table_path(name):
    return f"{tables_root(TARGET_BASE, TARGET_SCHEMA)}/{name}"

def is_delta(path):
    return DeltaTable.is_deltatable(path, storage_options=storage_options())

WRITE_OPTS = {"schema_mode": "merge"}
if ROW_GROUP_ROWS:
    from deltalake import WriterProperties
    WRITE_OPTS["writer_properties"] = WriterProperties(max_row_group_size=ROW_GROUP_ROWS)

TS = pl.Datetime("us", "UTC")
RUN_ID = str(uuid.uuid4())
def utcnow():
    return datetime.now(timezone.utc)

## Rules file
One JSON file (`RULES_FILE`, in the target Lakehouse's Files) with an entry per bronze table:

```json
{
  "tables": {
    "customers_raw": {
      "target_table": "customers",
      "duplicates": {"keys": ["customer_id"], "keep": "latest"},
      "cleanse": {
        "customer_name": ["trim", "collapse_spaces", "title_case"],
        "country": ["trim", {"map": {"UK": "United Kingdom", "USA": "United States"}}]
      },
      "allowed_values": {
        "country": {"values": ["United Kingdom", "Ireland", "France"], "on_fail": "reject"},
        "customer_name": {"allow_null": false, "on_fail": "reject"}
      }
    }
  }
}
```

- **target_table**: optional; default is the bronze name without `_raw`.
- **duplicates**: `keys` = column(s) that identify a row. `keep`: `latest` (newest bronze load wins, and updates the silver row), `first` (the first version is kept; later ones are ignored) or `reject` (keys repeated within a batch are all rejected). Rows with a null key are rejected. Without `duplicates`, rows are appended.
- **cleanse**: rules applied in order. Text: `trim`, `upper`, `lower`, `title_case`, `collapse_spaces`, `remove_control_chars`, `null_if_empty`, `{"replace": {"pattern": "regex", "with": "text"}}`, `{"map": {"from": "to"}}`. Numbers: `abs`, `{"round": 2}`. Any type: `{"fill_null": value}`.
- **allowed_values**: per column, any of `values` (list), `min`, `max`, `allow_null` (default true), `ignore_case` (text, default false). `on_fail`: `reject` (row goes to the rejects table), `null`, or `default` (with `"default": value`). Checked after cleansing.

In [ ]:
STRING_RULES = {"trim", "upper", "lower", "title_case", "collapse_spaces",
                "remove_control_chars", "null_if_empty", "replace", "map"}
NUMERIC_RULES = {"abs", "round"}
ANY_RULES = {"fill_null"}
ALLOWED_KEYS = {"values", "min", "max", "allow_null", "ignore_case", "on_fail", "default"}

def rule_name(rule):
    if isinstance(rule, str):
        return rule
    if isinstance(rule, dict) and len(rule) == 1:
        return next(iter(rule))
    raise ValueError(f"cleanse rule must be a name or a one-key object, got {rule!r}")

def load_rules():
    path = f"{TARGET_BASE}/Files/{RULES_FILE}"
    try:
        text = notebookutils.fs.head(path, 10_000_000)
    except Exception as e:
        raise FileNotFoundError(f"Rules file not found: Files/{RULES_FILE} in {TARGET_LAKEHOUSE}") from e
    spec = json.loads(text)
    tables = spec.get("tables")
    if not isinstance(tables, dict) or not tables:
        raise ValueError(f"{RULES_FILE}: needs a non-empty 'tables' object")
    for src, t in tables.items():
        where = f"{RULES_FILE} › {src}"
        unknown = set(t) - {"target_table", "duplicates", "cleanse", "allowed_values"}
        if unknown:
            raise ValueError(f"{where}: unknown setting(s) {sorted(unknown)}")
        d = t.get("duplicates")
        if d is not None:
            if not d.get("keys"):
                raise ValueError(f"{where}: duplicates needs 'keys'")
            if d.get("keep", "latest") not in ("latest", "first", "reject"):
                raise ValueError(f"{where}: duplicates.keep must be latest, first or reject")
        for col, rules in t.get("cleanse", {}).items():
            for r in rules:
                name = rule_name(r)
                if name not in STRING_RULES | NUMERIC_RULES | ANY_RULES:
                    raise ValueError(f"{where}: column '{col}': unknown cleanse rule '{name}'")
        for col, a in t.get("allowed_values", {}).items():
            bad = set(a) - ALLOWED_KEYS
            if bad:
                raise ValueError(f"{where}: allowed_values '{col}': unknown setting(s) {sorted(bad)}")
            on_fail = a.get("on_fail", "reject")
            if on_fail not in ("reject", "null", "default"):
                raise ValueError(f"{where}: allowed_values '{col}': on_fail must be reject, null or default")
            if on_fail == "default" and "default" not in a:
                raise ValueError(f"{where}: allowed_values '{col}': on_fail 'default' needs a 'default' value")
    return tables

def is_text(dtype):
    return dtype == pl.Utf8

def is_number(dtype):
    return dtype.is_numeric() or isinstance(dtype, pl.Decimal)

def to_value(v, dtype):
    """A JSON value as a Python value of the column's type (for min/max/values/defaults)."""
    if v is None:
        return None
    try:
        return _to_value(v, dtype)
    except Exception:
        hint = " (use YYYY-MM-DD)" if dtype == pl.Date else (
               " (use YYYY-MM-DD HH:MM:SS)" if isinstance(dtype, pl.Datetime) else "")
        raise ValueError(f"value {v!r} can't be used for a {dtype} column{hint}") from None

def _to_value(v, dtype):
    if dtype == pl.Date:
        return date.fromisoformat(str(v))
    if isinstance(dtype, pl.Datetime):
        d = datetime.fromisoformat(str(v))
        return d if d.tzinfo else d.replace(tzinfo=timezone.utc)
    if isinstance(dtype, pl.Decimal):
        return Decimal(str(v))
    if dtype.is_integer():
        return int(v)
    if dtype.is_float():
        return float(v)
    if dtype == pl.Boolean:
        return bool(v)
    return str(v)

def lit(v, dtype):
    return pl.lit(to_value(v, dtype)).cast(dtype)

def cleanse_expr(col, rules, dtype, where):
    e = pl.col(col)
    for r in rules:
        name = rule_name(r)
        arg = r[name] if isinstance(r, dict) else None
        if name in STRING_RULES and not is_text(dtype):
            raise ValueError(f"{where}: '{name}' needs a text column, '{col}' is {dtype}")
        if name in NUMERIC_RULES and not is_number(dtype):
            raise ValueError(f"{where}: '{name}' needs a numeric column, '{col}' is {dtype}")
        if name == "trim":
            e = e.str.strip_chars()
        elif name == "upper":
            e = e.str.to_uppercase()
        elif name == "lower":
            e = e.str.to_lowercase()
        elif name == "title_case":
            e = e.str.to_titlecase()
        elif name == "collapse_spaces":
            e = e.str.replace_all(r"\s+", " ")
        elif name == "remove_control_chars":
            e = e.str.replace_all(r"\p{C}", "")
        elif name == "null_if_empty":
            e = pl.when(e.str.len_chars() == 0).then(None).otherwise(e)
        elif name == "replace":
            e = e.str.replace_all(arg["pattern"], arg.get("with", ""))
        elif name == "map":
            e = e.replace(arg)
        elif name == "abs":
            e = e.abs()
        elif name == "round":
            e = e.round(int(arg))
        elif name == "fill_null":
            try:
                e = e.fill_null(lit(arg, dtype))
            except ValueError as err:
                raise ValueError(f"{where}: column '{col}': fill_null {err}") from None
    return e.alias(col)

def allowed_fail_expr(col, a, dtype, where):
    """Boolean expression: True where the value breaks the column's allowed-values rule."""
    try:
        if a.get("on_fail") == "default":
            to_value(a["default"], dtype)
        return _allowed_fail_expr(col, a, dtype)
    except ValueError as e:
        raise ValueError(f"{where}: allowed_values '{col}': {e}") from None

def _allowed_fail_expr(col, a, dtype):
    c = pl.col(col)
    ok = pl.lit(True)
    if "values" in a:
        vals = [to_value(v, dtype) for v in a["values"]]
        if a.get("ignore_case") and is_text(dtype):
            ok = ok & c.str.to_lowercase().is_in([v.lower() for v in vals])
        else:
            ok = ok & c.is_in(vals)          # a plain list: no is_in ambiguity on any Polars version
    if "min" in a:
        ok = ok & (c >= lit(a["min"], dtype))
    if "max" in a:
        ok = ok & (c <= lit(a["max"], dtype))
    null_ok = a.get("allow_null", True)
    # a null passes (or fails) on allow_null alone; comparisons with null are null
    return pl.when(c.is_null()).then(pl.lit(not null_ok)).otherwise(~ok.fill_null(False))

def describe_rule(a):
    parts = []
    if "values" in a:
        parts.append(f"in {len(a['values'])} allowed values")
    if "min" in a:
        parts.append(f">= {a['min']}")
    if "max" in a:
        parts.append(f"<= {a['max']}")
    if not a.get("allow_null", True):
        parts.append("not null")
    return ", ".join(parts) or "allowed"

RULES = load_rules()

# Which tables this run
if TABLES:
    wanted = [t.strip() for t in TABLES.split(",") if t.strip()]
    unknown = [t for t in wanted if t not in RULES]
    if unknown:
        raise ValueError(f"TABLES names table(s) with no entry in {RULES_FILE}: {unknown}")
    RUN_TABLES = {t: RULES[t] for t in wanted}
else:
    RUN_TABLES = dict(RULES)

def target_name(src, t):
    return t.get("target_table") or re.sub(r"_raw$", "", src)

# targets must be unique
seen = {}
for src, t in RUN_TABLES.items():
    tgt = target_name(src, t)
    if tgt in seen:
        raise ValueError(f"{seen[tgt]} and {src} both write to silver table {tgt}")
    seen[tgt] = src

# bronze tables with no rules (reported, not processed)
NO_RULES = []
if not TABLES:
    try:
        bronze = sorted(f.name for f in notebookutils.fs.ls(tables_root(SOURCE_BASE, SOURCE_SCHEMA)) if f.isDir)
    except Exception:
        bronze = []
    NO_RULES = [b for b in bronze if b not in RULES and b != "load_log"]

for src, t in RUN_TABLES.items():
    print(f"{src:<22} -> {target_name(src, t)}")
for b in NO_RULES:
    print(f"{b:<22} not in {RULES_FILE}; skipped")

## Process a table

In [ ]:
def processed_files(src, tgt):
    """Bronze source files already processed from this bronze table into this silver table."""
    p = target_table_path(PROCESSED_TABLE)
    if not is_delta(p):
        return []
    return (pl.scan_delta(p, storage_options=storage_options())
              .filter((pl.col("source_table") == src) & (pl.col("target_table") == tgt))
              .select("source_file").unique().collect()
              .to_series().to_list())

def row_json(df):
    """Each row as a JSON string, every value as text."""
    return df.select(pl.struct(pl.all().cast(pl.Utf8)).struct.json_encode().alias("row_json"))

def reject_records(original, marked, column, rule, reason, src, tgt):
    """Rows for the rejects table. `marked` has _rn (row number in the batch) and
    optionally _value (the failing value); the original bronze row is joined back by _rn,
    so each value stays with its own row."""
    if marked.height == 0:
        return None
    if "_value" not in marked.columns:
        marked = marked.with_columns(pl.lit(None, dtype=pl.Utf8).alias("_value"))
    rows = marked.select("_rn", pl.col("_value").cast(pl.Utf8)).join(original, on="_rn", how="left")
    return pl.DataFrame({
        "source_table": [src] * rows.height,
        "target_table": [tgt] * rows.height,
        "source_file": rows["_source_file"].cast(pl.Utf8),
        "column": [column] * rows.height,
        "rule": [rule] * rows.height,
        "value": rows["_value"],
        "reason": [reason] * rows.height,
        "row_json": row_json(rows.drop("_rn", "_value"))["row_json"],
    })

def merge_into(target, source, keys, keep):
    """Upsert df into the silver table on the key columns.
    Uses delta-rs streamed execution where available, which keeps memory down for large batches."""
    import inspect
    dt = DeltaTable(target, storage_options=storage_options())
    params = inspect.signature(dt.merge).parameters
    kwargs = {"predicate": " AND ".join(f't."{k}" = s."{k}"' for k in keys),
              "source_alias": "s", "target_alias": "t"}
    if "streamed_exec" in params:
        kwargs["streamed_exec"] = True
    if "writer_properties" in params and "writer_properties" in WRITE_OPTS:
        kwargs["writer_properties"] = WRITE_OPTS["writer_properties"]
    m = dt.merge(source, **kwargs)
    if keep == "first":
        m = m.when_not_matched_insert_all()                 # existing keys keep their first version
    else:
        m = m.when_matched_update_all().when_not_matched_insert_all()
    return m.execute()

def pending_files(lf):
    """New bronze files with their row counts, oldest load first."""
    return (lf.group_by("_source_file")
              .agg(pl.len().alias("rows"), pl.col("_load_ts").min().alias("loaded"))
              .sort(["loaded", "_source_file"])
              .collect())

def make_batches(files):
    """Group files (in load order) into batches of about BATCH_ROWS rows. A file is never split."""
    batches, cur, n = [], [], 0
    for f, rows in files.select("_source_file", "rows").iter_rows():
        if cur and n + rows > BATCH_ROWS:
            batches.append(cur)
            cur, n = [], 0
        cur.append(f)
        n += rows
    if cur:
        batches.append(cur)
    return batches

def apply_rules(df, t, prepared, src, tgt, counts, rejects):
    """Cleanse, check allowed values and remove duplicates for one batch. Returns the silver rows."""
    cleanse_exprs, allowed, dup, keys = prepared
    original = df
    # cleanse
    if cleanse_exprs:
        df = df.with_columns(cleanse_exprs)
    # allowed values
    reject_rn = []
    for col, (a, fail) in allowed.items():
        bad = df.filter(fail)
        if bad.height == 0:
            continue
        on_fail = a.get("on_fail", "reject")
        if on_fail == "reject":
            rejects.append(reject_records(original, bad.select("_rn", pl.col(col).alias("_value")), col,
                                          describe_rule(a), f"{col} not allowed ({describe_rule(a)})", src, tgt))
            reject_rn.append(bad.select("_rn"))
        else:
            new = pl.lit(None).cast(df.schema[col]) if on_fail == "null" else lit(a["default"], df.schema[col])
            df = df.with_columns(pl.when(fail).then(new).otherwise(pl.col(col)).alias(col))
            counts["values_replaced"] += bad.height
            print(f"[{src}] {bad.height:,} value(s) in {col} set to {on_fail}")
    if reject_rn:
        bad_rn = pl.concat(reject_rn).unique()
        df = df.join(bad_rn, on="_rn", how="anti")
        counts["rows_rejected"] += bad_rn.height
    # duplicates on the key columns
    if dup:
        null_key = df.filter(pl.any_horizontal([pl.col(k).is_null() for k in keys]))
        if null_key.height:
            rejects.append(reject_records(original, null_key.select("_rn"), ",".join(keys),
                                          "key not null", "null in key column(s)", src, tgt))
            df = df.join(null_key.select("_rn"), on="_rn", how="anti")
            counts["rows_rejected"] += null_key.height
        keep = dup.get("keep", "latest")
        # Cheap check first: if every key is unique there's nothing to resolve.
        # Otherwise find just the repeated keys and resolve only those rows.
        n_keys = (df[keys[0]].n_unique() if len(keys) == 1
                  else df.select(pl.struct(keys).n_unique()).item())
        if n_keys < df.height:
            dup_keys = collect_streaming(df.lazy().group_by(keys).len().filter(pl.col("len") > 1)).select(keys)
            dups = df.join(dup_keys, on=keys, how="semi")      # rows whose key repeats
            df = df.join(dup_keys, on=keys, how="anti")        # rows with a unique key
            if keep == "reject":
                rejects.append(reject_records(original, dups.select("_rn", pl.concat_str(
                    [pl.col(k).cast(pl.Utf8) for k in keys], separator="|").alias("_value")),
                    ",".join(keys), "unique key", "duplicate key in batch", src, tgt))
                counts["rows_rejected"] += dups.height
            else:
                winners = (dups.sort(["_load_ts", "_source_file", "_rn"])
                               .unique(subset=keys, keep="last" if keep == "latest" else "first",
                                       maintain_order=True))
                counts["duplicates_removed"] += dups.height - winners.height
                df = pl.concat([df, winners], how="vertical")
                del winners
            del dups, dup_keys
    del original
    # lineage: bronze _source_file/_load_ts are kept; _silver_updated_utc links to the run log
    return df.drop("_rn").with_columns(pl.lit(utcnow()).cast(TS).alias("_silver_updated_utc"))

def process_table(src, t):
    tgt = target_name(src, t)
    started = utcnow()
    counts = {"rows_read": 0, "rows_rejected": 0, "values_replaced": 0, "duplicates_removed": 0,
              "rows_out": 0, "rows_inserted": 0, "rows_updated": 0, "rows_appended": 0}
    result = {"source_table": src, "target_table": tgt, "status": "", "files": 0, "file_list": None,
              "batches": 0, "started_utc": started, "finished_utc": None, "seconds": 0.0, "error": None}
    rejects, processed, statuses = [], [], []
    try:
        bronze_path = source_table_path(src)
        if not is_delta(bronze_path):
            raise ValueError(f"bronze table {src} not found in {SOURCE_LAKEHOUSE}")
        where = f"{RULES_FILE} › {src}"

        # 1. check the rules against the bronze table's columns and types (before reading any rows)
        lf = pl.scan_delta(bronze_path, storage_options=storage_options())
        try:
            schema = dict(lf.collect_schema())
        except AttributeError:
            schema = dict(lf.schema)            # older Polars
        dup = t.get("duplicates")
        keys = dup["keys"] if dup else []
        named = set(t.get("cleanse", {})) | set(t.get("allowed_values", {})) | set(keys)
        missing = sorted(c for c in named if c not in schema)
        if missing:
            raise ValueError(f"{where}: column(s) not in {src}: {missing}")
        prepared = (
            [cleanse_expr(c, rules, schema[c], where) for c, rules in t.get("cleanse", {}).items()],
            {c: (a, allowed_fail_expr(c, a, schema[c], where)) for c, a in t.get("allowed_values", {}).items()},
            dup, keys)

        # 2. new bronze files (all files on a full reload), oldest first, grouped into batches
        if not FULL_RELOAD:
            done = processed_files(src, tgt)
            if done:
                lf = lf.filter(~pl.col("_source_file").is_in(done))
        files = pending_files(lf)
        result["files"] = files.height
        result["file_list"] = json.dumps(files["_source_file"].to_list())
        if files.height == 0:
            result["status"] = "no new data"
            return result | counts, rejects, processed
        batches = make_batches(files)
        result["batches"] = len(batches)
        target = target_table_path(tgt)

        # 3. each batch: rules, then one Delta commit
        for i, batch_files in enumerate(batches, 1):
            df = lf.filter(pl.col("_source_file").is_in(batch_files)).collect().with_row_index("_rn")
            counts["rows_read"] += df.height
            batch_rejects = []
            df = apply_rules(df, t, prepared, src, tgt, counts, batch_rejects)
            counts["rows_out"] += df.height
            gc.collect()
            first_write = (i == 1 and FULL_RELOAD) or not (DRY_RUN or is_delta(target))
            if DRY_RUN:
                status = "dry run"
            elif first_write:
                df.write_delta(target, mode="overwrite", storage_options=storage_options(),
                               delta_write_options={**WRITE_OPTS, "schema_mode": "overwrite"})
                counts["rows_inserted"] += df.height
                status = "reloaded" if FULL_RELOAD else "created"
            elif dup:
                # hand delta-rs an Arrow copy and release the Polars frame first, so both aren't held
                source = df.to_arrow()
                df = None
                gc.collect()
                metrics = merge_into(target, source, keys, dup.get("keep", "latest"))
                del source
                counts["rows_inserted"] += metrics.get("num_target_rows_inserted", 0)
                counts["rows_updated"] += metrics.get("num_target_rows_updated", 0)
                status = "merged"
            else:
                df.write_delta(target, mode="append", storage_options=storage_options(),
                               delta_write_options=WRITE_OPTS)
                counts["rows_appended"] += df.height
                status = "appended"
            statuses.append(status)
            # committed: these files (and their rejects) are done even if a later batch fails
            rejects += batch_rejects
            processed += [{"source_table": src, "target_table": tgt, "source_file": f} for f in batch_files]
            del df
            gc.collect()
            if len(batches) > 1:
                print(f"[{src}] batch {i}/{len(batches)}: {len(batch_files)} file(s) done")

        result["status"] = statuses[0] if statuses[0] in ("dry run", "reloaded", "created") else statuses[-1]
        written = (f"{counts['rows_out']:,} rows would be written" if DRY_RUN else
                   f"{counts['rows_inserted']:,} inserted, {counts['rows_updated']:,} updated, "
                   f"{counts['rows_appended']:,} appended")
        print(f"[{src}] {counts['rows_read']:,} read, {counts['rows_rejected']:,} rejected, "
              f"{counts['duplicates_removed']:,} duplicates removed -> {tgt}: {written} ({result['status']})")
    except Exception as e:
        result["status"] = "failed"
        result["error"] = str(e).splitlines()[0][:1000]
        done_files = len(processed)
        print(f"[{src}] FAILED: {result['error']}"
              + (f" ({done_files} file(s) in earlier batches were committed)" if done_files else ""))
    finally:
        result["finished_utc"] = utcnow()
        result["seconds"] = round((result["finished_utc"] - started).total_seconds(), 1)
        gc.collect()
    # rejects and processed files of committed batches are kept, even if a later batch failed
    return result | counts, [r for r in rejects if r is not None], processed

## Run

In [ ]:
run_started_utc = utcnow()
workers = max(1, min(PARALLEL_TABLES, len(RUN_TABLES) or 1))
if workers == 1:
    outcomes = [process_table(s, t) for s, t in RUN_TABLES.items()]
else:
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="table") as pool:
        outcomes = list(pool.map(lambda kv: process_table(*kv), RUN_TABLES.items()))
run_finished_utc = utcnow()

results = [o[0] for o in outcomes]
for b in NO_RULES:
    results.append({"source_table": b, "target_table": None, "status": "no rules", "files": 0,
                    "file_list": None, "batches": 0, "rows_read": 0, "rows_rejected": 0,
                    "values_replaced": 0, "duplicates_removed": 0, "rows_out": 0, "rows_inserted": 0,
                    "rows_updated": 0, "rows_appended": 0, "started_utc": None, "finished_utc": None,
                    "seconds": 0.0, "error": None})
summary = pl.DataFrame(results, schema={
    "source_table": pl.Utf8, "target_table": pl.Utf8, "status": pl.Utf8, "files": pl.Int64,
    "file_list": pl.Utf8, "batches": pl.Int64, "rows_read": pl.Int64, "rows_rejected": pl.Int64,
    "values_replaced": pl.Int64, "duplicates_removed": pl.Int64, "rows_out": pl.Int64, "rows_inserted": pl.Int64, "rows_updated": pl.Int64,
    "rows_appended": pl.Int64, "started_utc": TS, "finished_utc": TS, "seconds": pl.Float64,
    "error": pl.Utf8})
print(f"Run finished in {(run_finished_utc - run_started_utc).total_seconds():,.0f}s "
      f"with {workers} table(s) at a time")
summary.select("source_table", "target_table", "status", "files", "rows_read", "rows_rejected",
               "duplicates_removed", "rows_out", "rows_inserted", "rows_updated", "rows_appended", "seconds", "error")

## Record rejects, processed files and the run log
Written after all tables finish, in one commit each. Nothing from a failed table is marked as processed, so its files are picked up again next run. Dry runs write nothing.

In [ ]:
WRITE_ERRORS = []

def append(table, df):
    if df is None or df.height == 0:
        return
    try:
        df.write_delta(target_table_path(table), mode="append",
                       storage_options=storage_options(), delta_write_options={"schema_mode": "merge"})
    except Exception as e:
        WRITE_ERRORS.append(f"{table}: {str(e).splitlines()[0][:300]}")
        print(f"WARNING: could not write {table}: {WRITE_ERRORS[-1]}")

now = pl.lit(utcnow()).cast(TS)
if DRY_RUN:
    n = sum(r.height for o in outcomes for r in o[1])
    print(f"Dry run: nothing written ({n:,} reject record(s) would be written)")
else:
    rej = [r for o in outcomes for r in o[1]]
    if rej:
        append(REJECTS_TABLE, pl.concat(rej, how="vertical_relaxed").with_columns(
            pl.lit(RUN_ID).alias("run_id"), now.alias("rejected_utc")))
    done = [p for o in outcomes for p in o[2]]
    if done:
        append(PROCESSED_TABLE, pl.DataFrame(done, schema={"source_table": pl.Utf8, "target_table": pl.Utf8, "source_file": pl.Utf8})
               .with_columns(pl.lit(RUN_ID).alias("run_id"), now.alias("processed_utc")))
    if LOG_TABLE:
        try:
            ctx = dict(notebookutils.runtime.context)
        except Exception:
            ctx = {}
        append(LOG_TABLE, summary.with_columns(
            pl.lit(RUN_ID).alias("run_id"),
            pl.lit(run_started_utc).cast(TS).alias("run_started_utc"),
            pl.lit(run_finished_utc).cast(TS).alias("run_finished_utc"),
            pl.lit(FULL_RELOAD).alias("full_reload"),
            pl.lit(SOURCE_LAKEHOUSE).alias("source_lakehouse"),
            pl.lit(TARGET_LAKEHOUSE).alias("target_lakehouse"),
            pl.lit(RULES_FILE).alias("rules_file"),
            pl.lit(str(ctx.get("currentNotebookName") or ""), dtype=pl.Utf8).alias("notebook_name"),
            pl.lit(str(ctx.get("isForPipeline", "")).lower() == "true").alias("triggered_by_pipeline"),
            pl.lit(pl.__version__).alias("polars_version"),
            pl.lit(deltalake.__version__).alias("deltalake_version"),
        ))
    print(f"Recorded run {RUN_ID}")

## Maintenance (optional)
Merges rewrite files, so compact the silver tables periodically.

In [ ]:
RUN_MAINTENANCE = False

if RUN_MAINTENANCE:
    for src, t in RUN_TABLES.items():
        p = target_table_path(target_name(src, t))
        if is_delta(p):
            dt = DeltaTable(p, storage_options=storage_options())
            print(target_name(src, t), dt.optimize.compact())
            dt.vacuum(retention_hours=168, dry_run=False, enforce_retention_duration=True)

In [ ]:
# Fail the run if any table (or a record write) failed, so a pipeline marks it failed
failed = summary.filter(pl.col("status") == "failed")
problems = [f"{r['source_table']}: {r['error']}" for r in failed.iter_rows(named=True)] + WRITE_ERRORS
if problems:
    raise RuntimeError(f"{len(problems)} problem(s): " + "; ".join(problems))
notebookutils.notebook.exit(json.dumps({
    "run_id": RUN_ID,
    "tables": json.loads(summary.drop("started_utc", "finished_utc", "file_list").write_json()),
}))